# Split KV Decode Attention Kernel

A Triton implementation of split-KV decode attention, layout-compatible with
FlashInfer's paged KV cache, characterized against FlashInfer and a naive
single-block Triton kernel.

**Runtime:** Google Colab GPU (T4 or A100). Use *Runtime → Change runtime type → GPU*.

---
| Section | What it does |
|---------|-------------|
| 0 | Install dependencies |
| 1 | Clone repo and set up paths |
| 2 | KV layout demo |
| 3 | Gate 1 — FlashInfer layout sanity |
| 4 | Kernel correctness (Gates 2 & 3) |
| 5 | Microbenchmark (quick sweep) |
| 6 | Plots |
| 7 | E2E smoke test (Llama 3.2 1B) |

## 0  Install dependencies

Detects the CUDA and torch versions Colab provides, then installs the matching FlashInfer wheel.

In [ ]:
import subprocess, sys

def run(cmd): subprocess.run(cmd, shell=True, check=True)

# Detect torch version for the correct FlashInfer wheel
import torch
cuda_ver = torch.version.cuda.replace('.', '')   # e.g. '121'
torch_ver = '.'.join(torch.__version__.split('.')[:2])  # e.g. '2.3'
fi_index = f"https://flashinfer.ai/whl/cu{cuda_ver}/torch{torch_ver}/"
print(f"CUDA {cuda_ver}  torch {torch_ver}")
print(f"FlashInfer index: {fi_index}")

run(f"pip install flashinfer==0.1.6 --extra-index-url {fi_index} -q")
run("pip install transformers==4.44.0 accelerate==0.32.0 pandas matplotlib scipy -q")
print("Done.")

## 1  Clone repo and configure paths

In [ ]:
import os, sys

REPO_URL = "https://github.com/prabhleenkaur/triton-flashinfer.git"  # update if fork
REPO_DIR = "/content/triton-flashinfer"

if not os.path.exists(REPO_DIR):
    run(f"git clone {REPO_URL} {REPO_DIR}")
else:
    print("Repo already cloned; pulling latest ...")
    run(f"git -C {REPO_DIR} pull")

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
sys.path.insert(0, os.path.join(REPO_DIR, "bench"))
os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU found — change runtime to GPU"
props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name}")
print(f"Compute capability: {props.major}.{props.minor}")
print(f"VRAM: {props.total_memory/1e9:.1f} GB")
if props.major < 8:
    print("NOTE: Part 2 (SGLang) requires sm_80+ (A100).  Part 1 runs on any GPU.")

## 2  KV layout demo

Shows how `synthesize` creates paged KV tensors and what each tensor contains.

In [ ]:
from layout import synthesize, PagedKVLayout

BATCH          = 2
CONTEXT_LENGTH = 64      # short for the demo
NUM_Q_HEADS    = 8
NUM_KV_HEADS   = 2
HEAD_DIM       = 64
PAGE_SIZE      = 16

q, kv_data, kv_indptr, kv_indices, kv_last_page_len, layout = synthesize(
    batch=BATCH, context_length=CONTEXT_LENGTH,
    num_q_heads=NUM_Q_HEADS, num_kv_heads=NUM_KV_HEADS,
    head_dim=HEAD_DIM, page_size=PAGE_SIZE,
    dtype=torch.float16, device='cuda',
)

print("Tensor shapes:")
print(f"  Q              : {tuple(q.shape)}       (batch, num_q_heads, head_dim)")
print(f"  kv_data        : {tuple(kv_data.shape)} (num_pages, 2, page_size, num_kv_heads, head_dim)")
print(f"  kv_indptr      : {tuple(kv_indptr.shape)}  — {kv_indptr.tolist()}")
print(f"  kv_indices     : {tuple(kv_indices.shape)} — {kv_indices.tolist()}")
print(f"  kv_last_page_len: {tuple(kv_last_page_len.shape)} — {kv_last_page_len.tolist()}")
print(f"\nPages per sequence: {kv_indptr[1].item() - kv_indptr[0].item()}")
print(f"Tokens in last page: {kv_last_page_len[0].item()} / {PAGE_SIZE}")

## 3  Gate 1 — FlashInfer layout sanity

Checks that FlashInfer's decode wrapper and our fp32 reference agree on the same paged tensors.
If they disagree, the block table is malformed — not the custom kernels.

In [ ]:
import flashinfer
from reference import reference_attention_paged

workspace = torch.empty(128 * 1024 * 1024, dtype=torch.uint8, device='cuda')
wrapper = flashinfer.BatchDecodeWithPagedKVCacheWrapper(workspace, "NHD")
wrapper.begin_forward(
    kv_indptr, kv_indices, kv_last_page_len,
    NUM_Q_HEADS, NUM_KV_HEADS, HEAD_DIM,
    PAGE_SIZE, data_type=q.dtype,
)
fi_out = wrapper.forward(q, kv_data)   # (batch, num_q_heads, head_dim)

ref = reference_attention_paged(q, kv_data, kv_indptr, kv_indices, kv_last_page_len)
ref = ref.to('cuda')

max_err = (fi_out.float() - ref.float()).abs().max().item()
print(f"FlashInfer vs fp32 reference — max_abs_err: {max_err:.4e}")
assert max_err < 1e-2, f"Gate 1 FAILED: {max_err:.4e}"
print("Gate 1 PASSED  ✓")

## 4  Kernel correctness — Gates 2 & 3

**Gate 2:** both kernels agree with the fp32 reference on the full sweep grid.  
**Gate 3:** edge cases and determinism.

In [ ]:
from kernel_naive import decode_attention_naive
from kernel_split_kv import decode_attention_split_kv

# Quick sweep (subset of the full grid for Colab speed)
CONTEXT_LENGTHS = [128, 512, 2048]
BATCH_SIZES     = [1, 4]
SPLIT_KVS       = [1, 2, 4, 8]
QH, KH, HD, PS  = 8, 2, 64, 16

MAX_ERR_FP16 = 1e-2
MEAN_ERR_FP16 = 1e-3

results = []
for ctx in CONTEXT_LENGTHS:
    for batch in BATCH_SIZES:
        q, kv_data, kv_indptr, kv_indices, kv_last, _ = synthesize(
            batch=batch, context_length=ctx,
            num_q_heads=QH, num_kv_heads=KH, head_dim=HD,
            page_size=PS, dtype=torch.float16, device='cuda',
        )
        ref = reference_attention_paged(q, kv_data, kv_indptr, kv_indices, kv_last).cuda()

        # Naive
        out_naive = decode_attention_naive(q, kv_data, kv_indptr, kv_indices, kv_last)
        err_naive = (out_naive.float() - ref).abs()

        for skv in SPLIT_KVS:
            out_split = decode_attention_split_kv(
                q, kv_data, kv_indptr, kv_indices, kv_last, split_kv=skv
            )
            err_split = (out_split.float() - ref).abs()
            status = "PASS" if (err_split.max() < MAX_ERR_FP16 and
                                err_split.mean() < MEAN_ERR_FP16) else "FAIL"
            results.append(dict(ctx=ctx, batch=batch, skv=skv,
                                naive_max=err_naive.max().item(),
                                split_max=err_split.max().item(),
                                status=status))

import pandas as pd
df_corr = pd.DataFrame(results)
print(df_corr.to_string(index=False))
assert (df_corr['status'] == 'PASS').all(), "Gate 2 FAILED"
print("\nGate 2 PASSED  ✓")

In [ ]:
print("Gate 3: edge cases")

def check(label, ctx, batch, q_heads, kv_heads, skv=4):
    q, kv_d, indptr, indices, last, _ = synthesize(
        batch=batch, context_length=ctx,
        num_q_heads=q_heads, num_kv_heads=kv_heads,
        head_dim=64, page_size=16, dtype=torch.float16, device='cuda'
    )
    ref = reference_attention_paged(q, kv_d, indptr, indices, last).cuda()
    out = decode_attention_split_kv(q, kv_d, indptr, indices, last, split_kv=skv)
    # Determinism: run twice, expect identical output
    out2 = decode_attention_split_kv(q, kv_d, indptr, indices, last, split_kv=skv)
    deterministic = out.equal(out2)
    err = (out.float() - ref).abs().max().item()
    ok = err < 1e-2 and deterministic
    status = 'PASS' if ok else 'FAIL'
    print(f"  {label:<45} max_err={err:.2e}  det={deterministic}  {status}")
    return ok

ok = True
ok &= check("ctx=1",                              1,   1, 8, 2)
ok &= check("ctx not divisible by page_size(17)", 17,  1, 8, 2)
ok &= check("ctx not divisible by SPLIT_KV(30)",  30,  1, 8, 2, skv=4)
ok &= check("batch=1",                            128, 1, 8, 2)
ok &= check("MHA (group_size=1)",                 128, 1, 4, 4)
ok &= check("MQA (group_size=num_q_heads=8)",     128, 1, 8, 1)

assert ok, "Gate 3 FAILED"
print("\nGate 3 PASSED  ✓")

## 5  Microbenchmark (quick sweep)

Runs a small subset of the full benchmark grid.  For the complete sweep use `bench/microbench.py`.

In [ ]:
from microbench import (
    run_sweep, QUICK_CONTEXT_LENGTHS, QUICK_BATCH_SIZES,
    QUICK_SPLIT_KVS, QUICK_IMPLS,
)

os.makedirs('results', exist_ok=True)
df_bench = run_sweep(
    context_lengths=QUICK_CONTEXT_LENGTHS,
    batch_sizes=QUICK_BATCH_SIZES,
    split_kvs=QUICK_SPLIT_KVS,
    implementations=QUICK_IMPLS,
    out_path='results/microbench_quick.csv',
)
df_bench

## 6  Plots

Run the full benchmark first (`microbench.py`) to get the complete CSV, then call `plot.py`.
The cells below generate plots from whatever CSV was produced in section 5.

In [ ]:
import sys
sys.path.insert(0, 'bench')
from plot import (
    load,
    plot1_latency_vs_split,
    plot2_latency_vs_ctx,
    plot3_bw_pct,
    plot4_roofline,
    plot5_speedup_heatmap,
    plot6_gap_to_flashinfer,
)
from pathlib import Path

out_dir = Path('results/plots')
out_dir.mkdir(parents=True, exist_ok=True)

df_plot = load('results/microbench_quick.csv')
print(f"Loaded {len(df_plot)} rows")

In [ ]:
from IPython.display import Image, display
import matplotlib
matplotlib.use('Agg')

plot1_latency_vs_split(df_plot, out_dir)
display(Image('results/plots/plot1_latency_vs_split.png'))

In [ ]:
plot2_latency_vs_ctx(df_plot, out_dir)
display(Image('results/plots/plot2_latency_vs_ctx.png'))

In [ ]:
plot3_bw_pct(df_plot, out_dir)
display(Image('results/plots/plot3_bw_pct.png'))

In [ ]:
# Pass your GPU's actual peak BW here — check torch.cuda.get_device_properties(0)
plot4_roofline(df_plot, out_dir,
               peak_compute_tflops=65.0,   # T4 fp16
               peak_bw_gb_s=300.0)         # T4 HBM
display(Image('results/plots/plot4_roofline.png'))

In [ ]:
plot5_speedup_heatmap(df_plot, out_dir)
display(Image('results/plots/plot5_speedup_heatmap.png'))

In [ ]:
plot6_gap_to_flashinfer(df_plot, out_dir)
display(Image('results/plots/plot6_gap_to_flashinfer.png'))

## 7  E2E smoke test — Llama 3.2 1B

Loads Llama 3.2 1B, runs the split-KV kernel on synthetic paged KV tensors with
dimensions matching the model config, and checks max error vs fp32 reference.

> **Requires:** a Hugging Face token for the gated `meta-llama/Llama-3.2-1B` repo.
> Set `HF_TOKEN` in Colab Secrets (key icon in the left sidebar) before running.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print("HF_TOKEN loaded from Colab Secrets")
except Exception:
    print("Set HF_TOKEN manually: os.environ['HF_TOKEN'] = '...'")

In [ ]:
from e2e_smoke import verify_kernel_on_paged_kv, load_model

model, tokenizer = load_model('cuda')
cfg = model.config
print(f"Model:  {cfg._name_or_path}")
print(f"Heads:  {cfg.num_attention_heads} Q / {cfg.num_key_value_heads} KV")
print(f"Head dim: {cfg.hidden_size // cfg.num_attention_heads}")

max_err = verify_kernel_on_paged_kv(
    model, context_length=256, batch=1, device='cuda'
)
print(f"\nE2E smoke test max_abs_err: {max_err:.4e}  — {'PASS' if max_err < 1e-2 else 'FAIL'}")

---
## Summary

| Step | Status |
|------|--------|
| Gate 1 — FlashInfer layout sanity | ✓ |
| Gate 2 — Kernel correctness sweep | ✓ |
| Gate 3 — Edge cases + determinism | ✓ |
| Microbenchmark CSV | `results/microbench_quick.csv` |
| Plots | `results/plots/` |
| E2E smoke test | ✓ |

For the **full benchmark** (6 context lengths × 3 batch sizes × 5 splits):
```bash
python bench/microbench.py --out results/microbench.csv
python bench/plot.py --csv results/microbench.csv
```

For **Part 2 (SGLang on A100)**, see `integration/README.md`.